# CSC2042S Assignment 2: Perceptron Image Classification

**Student Number:** `FFLADA001` &nbsp;&nbsp; **Name:** `Adam Foflonker`

**High-level pipeline:**
1. Data Processing
2. Multi-class Perceptron Implementation
3. Training
4. Normalization
5. Evaluation
6. RGB vs Grayscale Comparison
7. Stopping Criteria Comparison
---

## Imports

In [ ]:
import os
import random
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

import matplotlib.pyplot as plt
%matplotlib inline

from pathlib import Path

## Global Constants, State & Flags

In [ ]:
DATA_DIR_GRAYSCALE: Path = Path("data/grayscale")
DATA_DIR_RGB: Path = Path("data/rgb")

TEST_SIZE: float = 0.3
RANDOM_SEED: int = 42

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

EPOCHS: int = 50
PATIENCE: int = 10 # Was 3
LEARNING_RATE: float = 0.001 # Was 0.01

LABEL_COUNT: int = 10

# Mapping from label index to character name
LABEL_MAP: dict[int, str] = {
    0: "bart_simpson",
    1: "charles_montgomery_burns",
    2: "homer_simpson",
    3: "krusty_the_clown",
    4: "lisa_simpson",
    5: "marge_simpson",
    6: "milhouse_van_houten",
    7: "moe_szyslak",
    8: "ned_flanders",
    9: "principal_skinner"
}

LABEL_MAP_INV = {v: k for k, v in LABEL_MAP.items()}
LABEL_NAMES = list(LABEL_MAP_INV.keys())

### 1) Data Processing

In [ ]:
def load_dataset(base_path, mode = "grayscale"):
    X, y = [], []
    base_path = str(base_path)
    train_path = os.path.join(base_path, "train")

    if not os.path.exists(train_path):
        raise FileNotFoundError(f"Training folder not found at {train_path}! Please download the Simpsons-MNIST dataset and place it under the path: {base_path}")

    for label_name in sorted(os.listdir(train_path)):
        label_dir = os.path.join(train_path, label_name)
        if not os.path.isdir(label_dir):
            continue
        if label_name not in LABEL_MAP_INV:
            continue
        
        label = LABEL_MAP_INV[label_name]
        for file in sorted(os.listdir(label_dir)):
            img_path = os.path.join(label_dir, file)
            try:
                img = Image.open(img_path)
            except Exception as e:
                print(f"Warning: could not open image {img_path}: {e}")
                continue

            img = img.convert("L") if mode == "grayscale" else img.convert("RGB")
            arr = np.array(img).flatten().astype(np.float32)
            X.append(arr)
            y.append(label)

    return np.array(X), np.array(y)

# Load datasets
X_gray, y_gray = load_dataset(DATA_DIR_GRAYSCALE, mode = "grayscale")
X_rgb, y_rgb = load_dataset(DATA_DIR_RGB, mode = "rgb")

# Split into train and temp (validation + test)
X_train_gray, X_temp_gray, y_train_gray, y_temp_gray = train_test_split(X_gray, y_gray, test_size = TEST_SIZE, 
                                                                        random_state = RANDOM_SEED, stratify = y_gray)
X_train_rgb, X_temp_rgb, y_train_rgb, y_temp_rgb = train_test_split(X_rgb, y_rgb, test_size = TEST_SIZE, 
                                                                    random_state = RANDOM_SEED, stratify = y_rgb)

# Split temp halfway into validation and test sets
X_val_gray, X_test_gray, y_val_gray, y_test_gray = train_test_split(X_temp_gray, y_temp_gray, test_size = 0.5, 
                                                                    random_state = RANDOM_SEED, stratify = y_temp_gray)
X_val_rgb, X_test_rgb, y_val_rgb, y_test_rgb = train_test_split(X_temp_rgb, y_temp_rgb, test_size = 0.5, 
                                                                random_state = RANDOM_SEED, stratify = y_temp_rgb)

print("Grayscale set shapes:")
print(" Train:", X_train_gray.shape, y_train_gray.shape)
print(" Val:", X_val_gray.shape, y_val_gray.shape)
print(" Test:", X_test_gray.shape, y_test_gray.shape)
print()
print("RGB set shapes:")
print(" Train:", X_train_rgb.shape, y_train_rgb.shape)
print(" Val:", X_val_rgb.shape, y_val_rgb.shape)
print(" Test:", X_test_rgb.shape, y_test_rgb.shape)

### 2) Multi-class Perceptron Implementation

Using One-versus-rest classification.

In [ ]:
class BinaryPerceptron:
    def __init__(self, n_features, learning_rate = LEARNING_RATE):
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.lr = learning_rate

    def predict(self, X):
        # X: 1D feature vector
        linear_output = np.dot(X, self.weights) + self.bias
        return 1 if linear_output >= 0 else 0

    def update(self, X, y):
        # Single epoch update; iterate samples and apply perceptron rule
        for xi, target in zip(X, y):
            pred = self.predict(xi)
            update = self.lr * (target - pred)
            self.weights += update * xi
            self.bias += update


class MultiClassPerceptron:
    def __init__(self, n_classes, n_features, learning_rate = LEARNING_RATE):
        self.classes = n_classes
        # One-vs-rest
        self.models = [BinaryPerceptron(n_features, learning_rate) for _ in range(n_classes)]

    def predict(self, X):
        if X.ndim == 1:
            scores = [np.dot(X, m.weights) + m.bias for m in self.models]
            return int(np.argmax(scores))
        
        scores = np.vstack([np.dot(X, m.weights) + m.bias for m in self.models])  # (n_classes, n_samples)
        return np.argmax(scores, axis = 0)

    def update(self, X, y):
        for xi, target in zip(X, y):
            for i, model in enumerate(self.models):
                binary_label = 1 if target == i else 0
                pred = model.predict(xi)
                update = model.lr * (binary_label - pred)
                model.weights += update * xi
                model.bias += update

### 3) Training

In [ ]:
def train_perceptron(model, X_train, y_train, X_val, y_val, epochs = EPOCHS, stop_criteria = "epochs", patience = PATIENCE):
    """
    Train perceptron with support for:
    - Fixed number of epochs (stop_criteria = "epochs")
    - Early stopping based on validation accuracy (stop_criteria = "early_stop")

    patience: number of epochs with no improvement before stopping early.
    Restores best model weights if early stopping triggers.
    """
    
    history = { "train_acc": [], "val_acc": [] }
    best_val = -1.0
    patience_counter = 0
    best_weights = None

    for epoch in range(epochs):
        # Shuffle training data
        indices = np.arange(len(X_train))
        np.random.shuffle(indices)
        X_train, y_train = X_train[indices], y_train[indices]
        model.update(X_train, y_train)

        # Compute accuracies
        train_acc = np.mean([model.predict(xi) == yi for xi, yi in zip(X_train, y_train)])
        val_acc = np.mean([model.predict(xi) == yi for xi, yi in zip(X_val, y_val)])
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch + 1}/{epochs}, Train Acc: {train_acc:.3f}, Val Acc: {val_acc:.3f}")

        # Early stopping
        if stop_criteria == "early_stop":
            if val_acc > best_val:
                best_val = val_acc
                patience_counter = 0
                # Save best weights (deep copy)
                if hasattr(model, "models"):
                    best_weights = [(np.copy(m.weights), float(m.bias)) for m in model.models]
                else:
                    best_weights = (np.copy(model.weights), float(model.bias))
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping at epoch {epoch + 1} (best val = {best_val:.4f})")
                    # Restore best weights
                    if best_weights is not None:
                        if hasattr(model, "models"):
                            for m, (w, b) in zip(model.models, best_weights):
                                m.weights = np.copy(w)
                                m.bias = float(b)
                        else:
                            model.weights = np.copy(best_weights[0])
                            model.bias = float(best_weights[1])
                    break

    return history

### 4) Normalization

In [ ]:
def normalise_data(X, method = "none"):
    """
    Apply different normalisation strategies.
    method: "none", "minmax", or "zscore"
    
    NOTE: normalisation is applied per-feature across the dataset (axis = 0)
    so that each input dimension is scaled consistently across samples.
    """
    
    if method == "none":
        return X
    elif method == "minmax":
        X_min = X.min(axis = 0, keepdims = True)
        X_max = X.max(axis = 0, keepdims = True)
        
        return (X - X_min) / (X_max - X_min + 1e-8)
    elif method == "zscore":
        mean = X.mean(axis = 0, keepdims = True)
        std = X.std(axis = 0, keepdims = True) + 1e-8
        
        return (X - mean) / std
    else:
        raise ValueError(f"Unknown normalisation method: {method}")

### 5. Evaluation

In [ ]:

def evaluate_model(model, X_train, y_train, X_val, y_val, X_test, y_test, normalisation = "none", title_suffix = "", train = True):
    """
    Evaluate a model: train it and report metrics. 
    Returns a dict with history, val/test metrics and per-class metrics.
    """
    
    X_train_norm = normalise_data(X_train, normalisation)
    X_val_norm = normalise_data(X_val, normalisation)
    X_test_norm = normalise_data(X_test, normalisation)

    history = None
    if train:
        history = train_perceptron(model, X_train_norm, y_train, X_val_norm, y_val, stop_criteria = "epochs")

    # Predictions
    y_val_pred = [int(model.predict(xi)) for xi in X_val_norm]
    y_test_pred = [int(model.predict(xi)) for xi in X_test_norm]

    # Validation metrics
    val_acc = accuracy_score(y_val, y_val_pred)
    val_prec = precision_score(y_val, y_val_pred, average = "macro", zero_division = 0)
    val_rec = recall_score(y_val, y_val_pred, average = "macro", zero_division = 0)
    val_f1 = f1_score(y_val, y_val_pred, average = "macro", zero_division = 0)

    # Test metrics
    test_acc = accuracy_score(y_test, y_test_pred)
    test_prec = precision_score(y_test, y_test_pred, average = "macro", zero_division = 0)
    test_rec = recall_score(y_test, y_test_pred, average = "macro", zero_division = 0)
    test_f1 = f1_score(y_test, y_test_pred, average = "macro", zero_division = 0)

    print(f"Validation Results {title_suffix}: Acc = {val_acc:.3f}, Prec = {val_prec:.3f}, Rec = {val_rec:.3f}, F1 = {val_f1:.3f}")
    print(f"Test Results {title_suffix}: Acc = {test_acc:.3f}, Prec = {test_prec:.3f}, Rec = {test_rec:.3f}, F1 = {test_f1:.3f}")

    # Confusion matrix
    cm = confusion_matrix(y_test, y_test_pred, labels = list(range(LABEL_COUNT)))
    fig, ax = plt.subplots(figsize = (12, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix = cm, display_labels = [LABEL_MAP[i] for i in range(LABEL_COUNT)])
    disp.plot(ax = ax, xticks_rotation = 45)
    ax.set_xticklabels(ax.get_xticklabels(), rotation = 45, ha = "right")
    plt.title(f"Confusion Matrix (Test Set) {title_suffix}")
    plt.tight_layout()
    plt.show()

    # Per-class metrics
    prec_per_class = precision_score(y_test, y_test_pred, average = None, zero_division = 0)
    rec_per_class = recall_score(y_test, y_test_pred, average = None, zero_division = 0)
    f1_per_class = f1_score(y_test, y_test_pred, average = None, zero_division = 0)

    metrics_table = []
    for i in range(LABEL_COUNT):
        metrics_table.append({
            "class": LABEL_MAP[i],
            "precision": float(prec_per_class[i]),
            "recall": float(rec_per_class[i]),
            "f1": float(f1_per_class[i])
        })

    return { "history": history, 
            "val": { "acc": val_acc, "prec": val_prec, "rec": val_rec, "f1": val_f1 }, 
            "test": {"acc": test_acc, "prec": test_prec, "rec": test_rec, "f1": test_f1}, 
            "per_class": metrics_table }

def compare_rgb_vs_gray(X_train_gray, y_train_gray, X_val_gray, y_val_gray, X_test_gray, y_test_gray,
                        X_train_rgb, y_train_rgb, X_val_rgb, y_val_rgb, X_test_rgb, y_test_rgb,
                        normalisation = "zscore"):
    n_features_gray = X_train_gray.shape[1]
    model_gray = MultiClassPerceptron(LABEL_COUNT, n_features_gray, learning_rate = LEARNING_RATE)
    results_gray = evaluate_model(model_gray, X_train_gray, y_train_gray, X_val_gray, y_val_gray, X_test_gray, y_test_gray, 
                                  normalisation = normalisation, title_suffix = "(Grayscale)")

    n_features_rgb = X_train_rgb.shape[1]
    model_rgb = MultiClassPerceptron(LABEL_COUNT, n_features_rgb, learning_rate=LEARNING_RATE)
    results_rgb = evaluate_model(model_rgb, X_train_rgb, y_train_rgb, X_val_rgb, y_val_rgb, X_test_rgb, y_test_rgb, 
                                 normalisation = normalisation, title_suffix = "(RGB)")

    # Summary
    print("\n=== Overall Comparison ===")
    print("Grayscale Test:", results_gray["test"])
    print("RGB Test:", results_rgb["test"])

    print("\n=== Per-class Comparison ===")
    for i in range(LABEL_COUNT):
        g = results_gray["per_class"][i]
        r = results_rgb["per_class"][i]
        print(f"{g['class']:25s} | Gray: P = {g['precision']:.2f}, R = {g['recall']:.2f}, F1 = {g['f1']:.2f} || " 
              f"RGB: P = {r['precision']:.2f}, R = {r['recall']:.2f}, F1 = {r['f1']:.2f}")

    return results_gray, results_rgb

def compare_stopping_criteria(X_train, y_train, X_val, y_val, X_test, y_test, n_features, title_suffix = "", normalisation = "zscore"):
    """
    Compare perceptron training using fixed epochs vs early stopping.
    Returns two result dicts (fixed, early_stop).
    """
    
    # Fixed epochs
    model_fixed = MultiClassPerceptron(LABEL_COUNT, n_features, learning_rate = LEARNING_RATE)
    history_fixed = train_perceptron(model_fixed, X_train, y_train, X_val, y_val, stop_criteria = "epochs")
    results_fixed = evaluate_model(model_fixed, X_train, y_train, X_val, y_val, X_test, y_test,
                                   normalisation = normalisation, title_suffix = f"(Fixed epochs {title_suffix})", train = False)

    # Early stopping
    model_es = MultiClassPerceptron(LABEL_COUNT, n_features, learning_rate = LEARNING_RATE)
    history_es = train_perceptron(model_es, X_train, y_train, X_val, y_val, stop_criteria  ="early_stop")
    results_es = evaluate_model(model_es, X_train, y_train, X_val, y_val, X_test, y_test,
                                normalisation = normalisation, title_suffix = f"(Early stopping {title_suffix})", train = False)

    # Plotting
    fig, ax = plt.subplots(figsize = (10, 6))
    ax.plot(history_fixed["val_acc"], label = "Fixed epochs")
    ax.plot(history_es["val_acc"], label = "Early stopping")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation Accuracy")
    ax.set_title(f"Stopping criteria comparison {title_suffix}")
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    # Summary
    print(f"\n=== Overall Comparison {title_suffix} ===")
    print("Fixed epochs Test:", results_fixed["test"])
    print("Early stopping Test:", results_es["test"])

    print(f"\n=== Per-class Comparison {title_suffix} ===")
    for i in range(LABEL_COUNT):
        f = results_fixed["per_class"][i]
        e = results_es["per_class"][i]
        print(f"{f['class']:25s} | Fixed: P = {f['precision']:.2f}, R = {f['recall']:.2f}, F1 = {f['f1']:.2f} "
              f"|| EarlyStop: P = {e['precision']:.2f}, R = {e['recall']:.2f}, F1 = {e['f1']:.2f}")

    return results_fixed, results_es, history_fixed, history_es

### 6) RGB vs Grayscale Comparison

In [ ]:
results_gray, results_rgb = compare_rgb_vs_gray(X_train_gray, y_train_gray, X_val_gray, y_val_gray, X_test_gray, y_test_gray, 
                                                X_train_rgb, y_train_rgb, X_val_rgb, y_val_rgb, X_test_rgb, y_test_rgb)

### 7) Stopping Criteria Comparison

In [ ]:
n_features_gray = X_train_gray.shape[1]
n_features_rgb = X_train_rgb.shape[1]

print("Grayscale comparison\n")

results_gray_fixed, results_gray_es, history_gray, history_gray_es = compare_stopping_criteria(
    X_train_gray, y_train_gray, X_val_gray, y_val_gray, X_test_gray, y_test_gray,
    n_features= n_features_gray, title_suffix = "(Grayscale)", normalisation = "zscore")

print("\nRGB Comparison\n")

results_rgb_fixed, results_rgb_es, history_rgb, history_rgb_es = compare_stopping_criteria(
    X_train_rgb, y_train_rgb, X_val_rgb, y_val_rgb, X_test_rgb, y_test_rgb,
    n_features= n_features_rgb, title_suffix = "(RGB)", normalisation = "zscore")